# Démonstration du pipeline ETL `xyt-walkability`
Ce notebook présente les étapes principales du pipeline ETL pour le calcul de l’indice de marchabilité, avec explications et exemples de code pour chaque étape.

## 1. Importer les bibliothèques nécessaires
Nous importons les modules du package `xyt-walkability` ainsi que les bibliothèques courantes pour la manipulation et la visualisation de données géospatiales.
- `geopandas` : manipulation de données géographiques
- `pandas` : manipulation de données tabulaires
- `matplotlib` : visualisation
- `xyt.walkability` : pipeline ETL modulaire pour la marchabilité

In [1]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from xyt.walkability.prepare_network import prepare_network
from xyt.walkability.prepare_features import prepare_features
from xyt.walkability.filter_features import filter_features
from xyt.walkability.aggregate_index import aggregate_index

Matplotlib is building the font cache; this may take a moment.


ModuleNotFoundError: No module named 'xyt'

## 2. Charger et afficher un jeu de données exemple
Nous allons extraire et segmenter un réseau piéton pour une zone d'étude (ex : Genève) à l'aide de la fonction `prepare_network`.

In [ ]:
# Extraction et segmentation du réseau piéton (étape 0)
result = prepare_network(
    input_path="Canton de Genève, Switzerland",
    output_path="./demo_output/step-1",
    params={"crs": "EPSG:2056"}
)
edges = result['edges']
nodes = result['nodes']

# Afficher les premières lignes du réseau segmenté
edges.head()

## 3. Nettoyer et enrichir les données
Nous enrichissons les segments du réseau avec des attributs (largeur, type, pente, etc.) à l'aide de la fonction `prepare_features`.

In [ ]:
# Enrichissement des segments avec les features (étape 1)
features = prepare_features(
    input_path="./demo_output/step-1/network_edges.geojson",
    output_path="./demo_output/step-2",
    params={"attributs_info": "./data/input/attributs/attributs_info.xlsx"}
)

# Afficher les premières lignes des features enrichis
features.head()

## 4. Filtrer et valider les features
On peut filtrer ou corriger manuellement les features (par exemple, dans QGIS ou via un notebook), puis sauvegarder les données nettoyées avec `filter_features`.

In [ ]:
# Filtrage/validation des features (étape 2)
features_filtered = filter_features(
    input_path="./demo_output/step-2/features_enriched.geojson",
    output_path="./demo_output/step-2b",
    params={"formats": ["geojson", "parquet"]}
)

# Afficher un aperçu des features filtrés
features_filtered.head()

## 5. Agréger et visualiser l’indice de marchabilité
On agrège les features par zone (GIREC, carreaux) et on calcule l’indice de marchabilité avec `aggregate_index`. On peut ensuite visualiser les résultats sur une carte.

In [ ]:
# Agrégation et calcul de l’indice (étape 3)
result = aggregate_index(
    input_path="./demo_output/step-2b/features_filtered.geojson",
    output_path="./demo_output/step-3",
    params={
        "zones_girec": "./data/input/zones_girec.gpkg",
        "agglo_carreau": "./data/input/carreau200.gpkg"
    }
)
girec = result['girec']
carreau = result['carreau']

# Visualisation simple de l’indice sur les zones GIREC
fig, ax = plt.subplots(figsize=(8, 6))
girec.plot(column="indice_marchabilite", ax=ax, legend=True, cmap="viridis")
ax.set_title("Indice de marchabilité par zone GIREC")
plt.show()